In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import DataLoader, Dataset, ConcatDataset
import numpy as np
import os
import zipfile
from PIL import Image
from tqdm.notebook import tqdm # progress bars

# Check if GPU is available
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# For reproducibility
torch.manual_seed(42)
np.random.seed(42)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(42)
PACS_DOMAINS = ['art_painting', 'cartoon', 'photo', 'sketch']
PACS_CATEGORIES = ['dog', 'elephant', 'giraffe', 'guitar', 'horse', 'house', 'person']


#Experiment Hyperparameters
MODEL_CHOICE = 'resnet18'
PRETRAINED = True
NUM_EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 3e-5

#identify logs/models
print(f"Using device: {DEVICE}, Model: {MODEL_CHOICE}, Epochs: {NUM_EPOCHS}, Batch: {BATCH_SIZE}, LR: {LEARNING_RATE}")


Using device: cuda
Using device: cuda, Model: resnet18, Epochs: 10, Batch: 32, LR: 3e-05


---

### Cell 2: Download and Prepare PACS Dataset



In [2]:
import os
import zipfile
from google.colab import drive

# Configuration
GOOGLE_DRIVE_FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/PACS.zip"
COLAB_ZIP_NAME = "PACS.zip"
BASE_PACS_PATH = "kfold"


# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. Copy from Google Drive
os.system(f"cp '{GOOGLE_DRIVE_FILE_PATH}' {COLAB_ZIP_NAME}")

# 3. Unzip
with zipfile.ZipFile(COLAB_ZIP_NAME, 'r') as zip_ref:
    zip_ref.extractall(".") # Extract to current directory

print(f"Dataset base path is configured as: '{BASE_PACS_PATH}'")

Mounted at /content/drive
Dataset base path is configured as: 'kfold'


---

### Cell 3: PACS_Dataset Class and Transformations





In [3]:
from pathlib import Path

class PACS_Dataset(Dataset):
    def __init__(self, base_path, domain_name, categories, transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []
        for label_idx, category in enumerate(categories):
            category_path = os.path.join(base_path, domain_name, category)
            if not os.path.isdir(category_path):
                continue
            for img_file in os.listdir(category_path):
                if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.image_paths.append(os.path.join(category_path, img_file))
                    self.labels.append(label_idx)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

# Data Transformations
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# With Augmentation
tta_transforms = [
    # Orininal image
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    # HorizontalFlip
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    #Rotation
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomRotation(degrees=15),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
]

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

---

### Cell 4: Model Definition (ResNet 18)

In [4]:
def get_model(num_classes=len(PACS_CATEGORIES), pretrained=True):
    # Load the ResNet18
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)

    # Replace the final fully connected layer to match the number of target classes
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)

    return model
# Test model
try:
    test_model = get_model(num_classes=len(PACS_CATEGORIES))
    print("Model created successfully.")
except Exception as e:
    print(f"Error creating model: {e}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 137MB/s]

Model created successfully.


---

### Cell 5: Training and Evaluation Functions

In [5]:
# Train (train_one_epoch function remains the same)
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in tqdm(dataloader, desc="Train", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

# Evaluation
def eval_one_epoch_tta(model, dataloader, criterion, device, tta_transforms, epoch_desc="Evaluating (TTA)"):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0


    progress_bar = tqdm(dataloader, desc=epoch_desc, leave=False)

    with torch.no_grad():
        for inputs_pil_list, labels in progress_bar:
            labels = labels.to(device) # a tensor
            batch_outputs_for_tta_versions = []

            # Key TTA: Iterate over each TTA transform
            for t_transform in tta_transforms:
                # Apply the current TTA transform
                current_tta_batch_tensors = torch.stack([t_transform(pil_img) for pil_img in inputs_pil_list])
                current_tta_batch_tensors = current_tta_batch_tensors.to(device)

                outputs = model(current_tta_batch_tensors)
                batch_outputs_for_tta_versions.append(outputs)

            # Average the outputs
            avg_logits = torch.stack(batch_outputs_for_tta_versions).mean(dim=0)
            loss = criterion(avg_logits, labels)

            running_loss += loss.item() * labels.size(0)  # Accumulate loss

            probs = torch.softmax(avg_logits, dim=1)
            preds = probs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)


    epoch_loss = running_loss / total if total > 0 else 0.0
    epoch_acc = correct / total if total > 0 else 0.0
    return epoch_loss, epoch_acc


---

### Cell 6: Leave-One-Domain-Out Evaluation Pipeline

In [6]:
import json
import os
import shutil
import torch
from torch.utils.data import DataLoader

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define your result directory
RESULTS_DIR = "/content/drive/MyDrive/xAi_DG_TTA/new_results"
MODEL_DIR = os.path.join(RESULTS_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

# Collate function for PIL images (TTA)
def tta_collate_fn(batch):
    images = [item[0] for item in batch]
    labels = torch.tensor([item[1] for item in batch], dtype=torch.long)
    return images, labels

# Store best TTA accuracy and model paths
all_accuracies = []
best_model_paths = []

# Leave-One-Domain-Out training + TTA evaluation
for target_domain in PACS_DOMAINS:
    print(f"\n===== Target Domain = {target_domain} =====")
    source_domains = [d for d in PACS_DOMAINS if d != target_domain]

    train_datasets = [
        PACS_Dataset(BASE_PACS_PATH, domain, PACS_CATEGORIES, transform=train_transform)
        for domain in source_domains
    ]
    train_dataset = torch.utils.data.ConcatDataset(train_datasets)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

    val_dataset = PACS_Dataset(BASE_PACS_PATH, target_domain, PACS_CATEGORIES, transform=None)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=tta_collate_fn)

    model = get_model(len(PACS_CATEGORIES), PRETRAINED).to(DEVICE)
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

    best_val_acc = 0.0
    best_model_state = None

    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc = eval_one_epoch_tta(model, val_loader, criterion, DEVICE, tta_transforms)
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Acc: {train_acc:.3f} | Val Acc: {val_acc:.3f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict()

    model_path = os.path.join(MODEL_DIR, f"best_model_target_{target_domain}.pth")
    torch.save(best_model_state, model_path)

    all_accuracies.append(best_val_acc)
    best_model_paths.append(model_path)

# Summary statistics
if all_accuracies:
    average_acc = sum(all_accuracies) / len(all_accuracies)
    worst_acc = min(all_accuracies)

    print("\n===== Cross-Domain Evaluation Summary ======")
    for domain, acc in zip(PACS_DOMAINS, all_accuracies):
        print(f"{domain}: {acc:.3f}")
    print(f"\nAverage Accuracy: {average_acc:.3f}")
    print(f"Worst-case Accuracy: {worst_acc:.3f}")
else:
    print("\nNo accuracies recorded to summarize.")

# Save per-domain TTA result
result_json_path = os.path.join(RESULTS_DIR, "tta_results.json")
with open(result_json_path, "w") as f:
    json.dump({
        "type": "tta",
        "domains": PACS_DOMAINS,
        "accuracies": all_accuracies,
        "average": average_acc,
        "worst_case": worst_acc
    }, f, indent=4)

# Re-evaluate each saved model across all domains (TTA)
print("\nEvaluating all models across all domains with TTA...")
model_scores = []
for path in best_model_paths:
    model = get_model(len(PACS_CATEGORIES), PRETRAINED).to(DEVICE)
    model.load_state_dict(torch.load(path))
    model.eval()

    domain_scores = []
    for eval_domain in PACS_DOMAINS:
        eval_dataset = PACS_Dataset(BASE_PACS_PATH, eval_domain, PACS_CATEGORIES, transform=None)
        eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=tta_collate_fn)
        _, val_acc = eval_one_epoch_tta(model, eval_loader, criterion, DEVICE, tta_transforms)
        domain_scores.append(val_acc)

    avg_score = sum(domain_scores) / len(domain_scores)
    model_scores.append((path, avg_score))
    print(f"{os.path.basename(path)} | TTA Avg Accuracy: {avg_score:.3f}")

# Save best generalized model
best_model_path, best_avg = max(model_scores, key=lambda x: x[1])
final_best_path = os.path.join(MODEL_DIR, "best_generalized_model_tta.pth")
shutil.copy(best_model_path, final_best_path)

# Save meta info
meta_path = os.path.join(RESULTS_DIR, "best_generalized_model_info_tta.json")
with open(meta_path, "w") as f:
    json.dump({
        "source_model_path": best_model_path,
        "copied_to": final_best_path,
        "average_accuracy": best_avg
    }, f, indent=4)

print(f"\nBest generalized TTA model saved to: {final_best_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

===== Target Domain = art_painting =====


Train:   0%|          | 0/249 [00:00<?, ?it/s]

Evaluating (TTA):   0%|          | 0/64 [00:00<?, ?it/s]

Epoch 1/10 | Train Acc: 0.817 | Val Acc: 0.699


Train:   0%|          | 0/249 [00:00<?, ?it/s]

...

Evaluating (TTA):   0%|          | 0/64 [00:00<?, ?it/s]

Evaluating (TTA):   0%|          | 0/74 [00:00<?, ?it/s]

Evaluating (TTA):   0%|          | 0/53 [00:00<?, ?it/s]

Evaluating (TTA):   0%|          | 0/123 [00:00<?, ?it/s]

best_model_target_sketch.pth | TTA Avg Accuracy: 0.893

Best generalized TTA model saved to: /content/drive/MyDrive/xAi_DG_TTA/new_results/models/best_generalized_model_tta.pth
